# HP1α condensate intensity and of the surrounding nucleoplasm

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from aicsimageio import AICSImage

In [ ]:
HP1A_CHANNEL = 1  # channel order: DAPI, HP1α, (actin), pillars


def image_path(data_dir, coordinate_file_name):
    """
    Path of the z-stack that a coordinate file `<date>_HeLa_HP1aGFP_REP<n>_<time>_<height>_<pitch>umpitch_Z_stack<s>_nuc<k>_...`
    belongs to, and the condition and pitch labels used in the output.
    """

    parts = coordinate_file_name.split("_")
    date, time, condition = parts[0], parts[4], parts[5]

    if "Imaging" in condition:
        condition = "Imaging_dish"
        pitch = "control_dish"
        session_folder = f"{date}_{time}_{condition}_data_files"
    else:
        pitch = parts[6].replace("pitch", "")
        session_folder = f"{date}_{condition}_{parts[6]}_{time}_data_files"

    hour_folder = "6hours" if "6h" in time else "24hours"
    image_name = "_".join(parts[:9]) + ".ome.tif"

    return Path(data_dir) / hour_folder / condition / session_folder / image_name, condition, pitch


def to_voxels(physical_coords, scale, shape):
    """Convert physical coordinates (µm) to voxel indices of the full image, clipped to the image."""

    voxels = np.round(physical_coords / scale).astype(int)
    return np.clip(voxels, 0, [s - 1 for s in shape])


def nucleoplasm_values(hp1a, nucleus_voxels, condensate_voxels):
    """
    HP1α values of the nucleus voxels that are not in any condensate.

    The masks are built in the bounding box of the nucleus only; the selected voxels and
    their order are the same as for masks of the full image.
    """

    low = nucleus_voxels.min(axis=0)
    high = nucleus_voxels.max(axis=0) + 1
    box_shape = tuple(high - low)

    nucleus_mask = np.zeros(box_shape, dtype=bool)
    nucleus_mask[tuple((nucleus_voxels - low).T)] = True

    condensate_mask = np.zeros(box_shape, dtype=bool)
    for voxels in condensate_voxels:
        inside = np.all((voxels >= low) & (voxels < high), axis=1)  # outside the nucleus box is irrelevant
        condensate_mask[tuple((voxels[inside] - low).T)] = True

    box = hp1a[low[0] : high[0], low[1] : high[1], low[2] : high[2]]
    return box[nucleus_mask & ~condensate_mask]


def process_replicate(replicate, coordinate_dir, data_dir):
    """Return the rows (one per condensate) of one replicate."""

    physical = Path(coordinate_dir) / "physical_coordinates"
    condensate_files = sorted((physical / "HP1a_condensates").glob("*.npz"))

    # Group the nuclei by image stack (first 9 name parts), so that each image is read once
    stacks = {}
    for f in condensate_files:
        stacks.setdefault("_".join(f.name.split("_")[:9]), []).append(f)
    print(f"{replicate}: {len(condensate_files)} nuclei in {len(stacks)} stacks")

    rows = []
    for stack_files in stacks.values():
        parts = stack_files[0].name.split("_")
        path, condition, pitch = image_path(data_dir, stack_files[0].name)
        if not path.exists():
            print(f"  Image not found, skipped: {path}")
            continue

        image = AICSImage(path)
        hp1a = np.squeeze(image.data)[HP1A_CHANNEL]
        scale = np.array([image.physical_pixel_sizes.Z, image.physical_pixel_sizes.Y, image.physical_pixel_sizes.X])

        for condensate_file in stack_files:
            nucleus_file = (
                physical
                / "nuclei"
                / condensate_file.name.replace("_condensate_physical_coords.npz", "_physical_coords.npy")
            )
            if not nucleus_file.exists():
                continue

            condensates = np.load(condensate_file)
            condensate_voxels = {key: to_voxels(condensates[key], scale, hp1a.shape) for key in condensates.files}
            nucleus_voxels = to_voxels(np.load(nucleus_file), scale, hp1a.shape)

            scaffold = nucleoplasm_values(hp1a, nucleus_voxels, condensate_voxels.values())
            if scaffold.size:
                scaffold_mean, scaffold_min, scaffold_max = np.mean(scaffold), np.min(scaffold), np.max(scaffold)
            else:
                scaffold_mean = scaffold_min = scaffold_max = np.nan

            for key, voxels in condensate_voxels.items():
                values = hp1a[voxels[:, 0], voxels[:, 1], voxels[:, 2]]
                rows.append(
                    {
                        "Date": parts[0],
                        "Replicate": parts[3],
                        "Adherence_Time": parts[4],
                        "Condition": condition,
                        "Pillar_Step": pitch,
                        "Z_Stack": parts[8],
                        "Nuc_Nr": condensate_file.name.split("_")[9],
                        "condensate_id": key,
                        "Cond_Voxel_Count": condensates[key].shape[0],
                        "mean_intensity_raw": np.mean(values),
                        "mean_intensity_corrected": np.mean(values) - scaffold_mean,
                        "median_intensity_raw": np.median(values),
                        "median_intensity_corrected": np.median(values) - scaffold_mean,
                        "max_intensity": np.max(values),
                        "min_intensity": np.min(values),
                        "nuc_scaffold_mean": scaffold_mean,
                        "nuc_scaffold_min": scaffold_min,
                        "nuc_scaffold_max": scaffold_max,
                    }
                )

    return rows

## Settings

Paths are relative to this notebook.

In [ ]:
DATA = Path("../../data")

# Coordinate folder and image folder of each replicate
REPLICATES = {
    "REP1": (DATA / "coordinate_files_REP1", DATA / "REP1"),
    "REP2": (DATA / "coordinate_files_REP2", DATA / "REP2"),
    "REP3": (DATA / "coordinate_files_REP3", DATA / "REP3"),
}
OUTPUT_DIR = Path(".")

## Compute the intensities


In [ ]:
for replicate, (coordinate_dir, data_dir) in REPLICATES.items():
    rows = process_replicate(replicate, coordinate_dir, data_dir)
    output = OUTPUT_DIR / f"Master_Condensate_Analysis_{replicate}.csv"
    pd.DataFrame(rows).to_csv(output, index=False)
    print(f"Wrote {len(rows)} condensates to {output}")